# Data Quality — Add Checks Notebook

**Run this notebook to register your semantic models and the DAX expressions you want to validate.**

For each metric you want to track across multiple models:
1. Pick a **check name** — use the same name across all models (e.g., `"Total Sales"`)
2. Add one row per model with the workspace, dataset, DAX, and row-level fail threshold for that model
3. The job will compare all models against the configured baseline (`MODEL` row via `is_baseline`, or `STATIC` numeric baseline)

The table is safe to re-run — duplicate registrations won't create duplicate rows.

In [ ]:
# -- Configuration -------------------------------------------------------------
# Load shared settings from config.py in Lakehouse.
import sys
sys.path.append("/lakehouse/default/Files/code")

try:
    from config import LAKEHOUSE_NAME, SCHEMA_NAME, MAX_RETRY_ATTEMPTS, INITIAL_RETRY_DELAY, DAX_TIMEOUT_SECONDS, RUN_TABLE_MAINTENANCE, MAINTENANCE_DAY_UTC
except ImportError:
    # Fallback keeps the notebook runnable if config file is unavailable.
    LAKEHOUSE_NAME = "MyLakehouse"
    SCHEMA_NAME = "data_quality"
    MAX_RETRY_ATTEMPTS = 3
    INITIAL_RETRY_DELAY = 2
    DAX_TIMEOUT_SECONDS = 60
    RUN_TABLE_MAINTENANCE = False
    MAINTENANCE_DAY_UTC = 6

FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"

## How to Add Checks

Edit the `checks` list in the cell below. For each row, provide:

| Column | Example | Options |
|---|---|---|
| **check_name** | `"Total Sales"` | Must be identical across all models for that metric |
| **model_name** | `"Finance GAAP"` | Your label for this model |
| **workspace_id** | `"11111111-2222-3333-4444-555555555555"` | Fabric workspace GUID (stable identity) |
| **dataset_id** | `"aaaaaaaa-bbbb-cccc-dddd-eeeeeeeeeeee"` | Semantic model GUID (stable identity) |
| **workspace_name** | `"Finance Workspace"` | Fabric workspace display name |
| **dataset_name** | `"Finance Model"` | Semantic model display name |
| **dax_expression** | `'EVALUATE ROW("value", [Sales Amount])'` | DAX returning one number |
| **run_frequency** | `"ONCE_PER_DAY"` | `ONCE_PER_DAY` (default, max once daily) or `MULTIPLE_PER_DAY` (can run multiple times/day) |
| **fail_delta_pct_threshold** | `0.5` | Required row-level percent threshold. Example: `0.5` means fail when `|delta_pct| > 0.5` |

**run_frequency:**
- Use `ONCE_PER_DAY` for daily metrics (most common)
- Use `MULTIPLE_PER_DAY` if you need to validate this check multiple times per day
- A single `check_name` must use one frequency across all its model rows (mixed frequencies are rejected)

**fail_delta_pct_threshold:**
- Required for every row you add
- Stored at the `(check_name, workspace_id, dataset_id)` level
- Can differ across models inside the same `check_name`

In [ ]:
from pyspark.sql.functions import lit, trim, upper
import time

# -- EDIT THE LIST BELOW ------------------------------------------------------
#
#   (check_name,        model_name,      workspace_id,                          dataset_id,                            workspace_name,        dataset_name,         dax_expression,                                    run_frequency,        fail_delta_pct_threshold)
#
checks = [
    ("Total Sales",     "Finance",       "11111111-2222-3333-4444-555555555555", "aaaaaaaa-bbbb-cccc-dddd-eeeeeeeeeeee", "Finance Workspace",   "Finance Model",      'EVALUATE ROW("value", [Sales Amount])',          "ONCE_PER_DAY",       0.10),
    ("Total Sales",     "Sales AMER",    "66666666-7777-8888-9999-000000000000", "ffffffff-1111-2222-3333-444444444444", "Sales Workspace",     "Sales AMER Model",   'EVALUATE ROW("value", [Total Revenue])',         "ONCE_PER_DAY",       0.25),
    ("Total Sales",     "Sales EMEA",    "66666666-7777-8888-9999-000000000000", "55555555-6666-7777-8888-999999999999", "Sales Workspace",     "Sales EMEA Model",   'EVALUATE ROW("value", [Net Sales])',             "ONCE_PER_DAY",       0.25),

    ("Total Customers", "Finance",       "11111111-2222-3333-4444-555555555555", "aaaaaaaa-bbbb-cccc-dddd-eeeeeeeeeeee", "Finance Workspace",   "Finance Model",      'EVALUATE ROW("value", [Customer Count])',        "ONCE_PER_DAY",       0.50),
    ("Total Customers", "Sales AMER",    "66666666-7777-8888-9999-000000000000", "ffffffff-1111-2222-3333-444444444444", "Sales Workspace",     "Sales AMER Model",   'EVALUATE ROW("value", [Distinct Customers])',   "ONCE_PER_DAY",       0.50),
]
# -----------------------------------------------------------------------------

from pyspark.sql.types import StructType, StructField, StringType, DoubleType
schema = StructType([
    StructField("check_name",                StringType()),
    StructField("model_name",                StringType()),
    StructField("workspace_id",              StringType()),
    StructField("dataset_id",                StringType()),
    StructField("workspace_name",            StringType()),
    StructField("dataset_name",              StringType()),
    StructField("dax_expression",            StringType()),
    StructField("run_frequency",             StringType()),
    StructField("fail_delta_pct_threshold",  DoubleType()),
])

# New rows start as non-baseline; baseline policy is enforced after merge.
df = (
    spark.createDataFrame(checks, schema=schema)
    .withColumn("is_active",   lit(True))
    .withColumn("is_baseline", lit(False))
)

df = (
    df.withColumn("check_name",     trim("check_name"))
      .withColumn("model_name",     trim("model_name"))
      .withColumn("workspace_id",   trim("workspace_id"))
      .withColumn("dataset_id",     trim("dataset_id"))
      .withColumn("workspace_name", trim("workspace_name"))
      .withColumn("dataset_name",   trim("dataset_name"))
      .withColumn("dax_expression", trim("dax_expression"))
      .withColumn("run_frequency",  upper(trim("run_frequency")))
)

# Strict required-field validation
required_field_checks = {
    "check_name":     "check_name",
    "model_name":     "model_name",
    "workspace_id":   "workspace_id",
    "dataset_id":     "dataset_id",
    "workspace_name": "workspace_name",
    "dataset_name":   "dataset_name",
    "dax_expression": "dax_expression",
}

for col_name, label in required_field_checks.items():
    bad_rows = df.filter(f"{col_name} IS NULL OR {col_name} = ''").select("check_name", "model_name", "workspace_id", "dataset_id").toPandas()
    if len(bad_rows) > 0:
        raise ValueError(
            f"{label} is required and cannot be blank. Failing rows:\n"
            + bad_rows.head(20).to_string(index=False)
        )

bad_threshold_rows = df.filter(
    "fail_delta_pct_threshold IS NULL OR fail_delta_pct_threshold < 0"
 ).select("check_name", "model_name", "workspace_id", "dataset_id", "fail_delta_pct_threshold").toPandas()
if len(bad_threshold_rows) > 0:
    raise ValueError(
        "fail_delta_pct_threshold is required and must be >= 0 for every submitted row.\n"
        + bad_threshold_rows.head(20).to_string(index=False)
    )

# Strict run_frequency enum validation
invalid_freq_rows = df.filter(
    "run_frequency IS NULL OR run_frequency = '' OR run_frequency NOT IN ('ONCE_PER_DAY', 'MULTIPLE_PER_DAY')"
 ).select("check_name", "model_name", "run_frequency").toPandas()
if len(invalid_freq_rows) > 0:
    raise ValueError(
        "Invalid run_frequency detected. Allowed values: ONCE_PER_DAY, MULTIPLE_PER_DAY.\n"
        + invalid_freq_rows.head(20).to_string(index=False)
    )

# Prevent duplicate identity keys in the same submission
submission_dupes = df.groupBy("check_name", "workspace_id", "dataset_id").count().filter("count > 1").toPandas()
if len(submission_dupes) > 0:
    raise ValueError(
        "Submission contains duplicate identity keys (check_name, workspace_id, dataset_id).\n"
        + submission_dupes.head(20).to_string(index=False)
    )

# Enforce frequency consistency: a check_name cannot be submitted with mixed run_frequency values
new_freq_conflicts = df.groupBy("check_name").count().join(
    df.select("check_name", "run_frequency").dropDuplicates().groupBy("check_name").count().withColumnRenamed("count", "freq_count"),
    on="check_name",
    how="inner"
).filter("freq_count > 1").select("check_name").dropDuplicates().toPandas()

if len(new_freq_conflicts) > 0:
    raise ValueError(
        "Mixed run_frequency values were found for the same check_name in this submission: "
        + ", ".join(new_freq_conflicts["check_name"].tolist())
    )

# Enforce consistency with existing registry rows for the same check_name
df.select("check_name", "run_frequency").dropDuplicates().createOrReplaceTempView("_new_check_freq")
existing_conflicts = spark.sql(f"""
    SELECT n.check_name, n.run_frequency AS new_run_frequency, e.run_frequency AS existing_run_frequency
    FROM _new_check_freq n
    INNER JOIN (
        SELECT DISTINCT check_name, UPPER(TRIM(run_frequency)) AS run_frequency
        FROM {FULL_SCHEMA}.check_registry
    ) e
      ON n.check_name = e.check_name
    WHERE n.run_frequency <> e.run_frequency
""").toPandas()

if len(existing_conflicts) > 0:
    raise ValueError(
        "run_frequency conflict for existing check_name(s). A check_name must use one frequency only. "
        + "Conflicts: "
        + ", ".join(
            f"{row['check_name']} (new={row['new_run_frequency']}, existing={row['existing_run_frequency']})"
            for _, row in existing_conflicts.iterrows()
        )
    )

# Fail fast if target table already has duplicate identity keys (safety for concurrent-writer environments)
existing_dupe_count = spark.sql(f"""
    SELECT COUNT(*) AS dupe_count
    FROM (
        SELECT check_name, workspace_id, dataset_id
        FROM {FULL_SCHEMA}.check_registry
        GROUP BY check_name, workspace_id, dataset_id
        HAVING COUNT(*) > 1
    ) d
""").collect()[0]["dupe_count"]
if existing_dupe_count > 0:
    raise RuntimeError(
        "check_registry currently has duplicate identity keys. Resolve duplicates before running add checks."
    )

# Upsert - safe to re-run, with retry for optimistic-concurrency conflicts
# Identity key uses check_name + workspace_id + dataset_id (model_name is display attribute)
# is_baseline is NOT updated for matched rows so existing flag values are always preserved.
df.createOrReplaceTempView("_new_checks")

max_merge_attempts = 3
for attempt in range(1, max_merge_attempts + 1):
    try:
        spark.sql(f"""
            MERGE INTO {FULL_SCHEMA}.check_registry AS t
            USING _new_checks AS s
              ON t.check_name = s.check_name
             AND t.workspace_id = s.workspace_id
             AND t.dataset_id = s.dataset_id
            WHEN NOT MATCHED THEN INSERT *
            WHEN MATCHED THEN UPDATE SET
              t.model_name                = s.model_name,
              t.workspace_id              = s.workspace_id,
              t.dataset_id                = s.dataset_id,
              t.workspace_name            = s.workspace_name,
              t.dataset_name              = s.dataset_name,
              t.dax_expression            = s.dax_expression,
              t.run_frequency             = s.run_frequency,
              t.fail_delta_pct_threshold  = s.fail_delta_pct_threshold,
              t.is_active                 = s.is_active
        """)
        break
    except Exception as e:
        msg = str(e).lower()
        is_concurrency_conflict = (
            "concurrent" in msg
            or "conflict" in msg
            or "transaction" in msg
            or "modified by another operation" in msg
        )
        if attempt < max_merge_attempts and is_concurrency_conflict:
            delay = 2 ** attempt
            print(f"Merge concurrency conflict on attempt {attempt}/{max_merge_attempts}; retrying in {delay}s...")
            time.sleep(delay)
            continue
        raise RuntimeError(f"MERGE into check_registry failed: {str(e)[:400]}")

post_merge_dupe_count = spark.sql(f"""
    SELECT COUNT(*) AS dupe_count
    FROM (
        SELECT check_name, workspace_id, dataset_id
        FROM {FULL_SCHEMA}.check_registry
        GROUP BY check_name, workspace_id, dataset_id
        HAVING COUNT(*) > 1
    ) d
""").collect()[0]["dupe_count"]
if post_merge_dupe_count > 0:
    raise RuntimeError(
        "Post-merge integrity check failed: duplicate identity keys detected in check_registry. "
        "Resolve duplicates before continuing."
    )

# Ensure every check_name has a baseline configuration row.
spark.sql(f"""
    MERGE INTO {FULL_SCHEMA}.check_baseline_config c
    USING (SELECT DISTINCT check_name FROM {FULL_SCHEMA}.check_registry) s
      ON c.check_name = s.check_name
    WHEN NOT MATCHED THEN INSERT (check_name, baseline_mode, static_baseline_value, updated_at)
      VALUES (s.check_name, 'MODEL', NULL, current_timestamp())
""")

# Force policy: if a check has exactly one active row, that row must be baseline.
spark.sql(f"""
    UPDATE {FULL_SCHEMA}.check_registry
    SET is_baseline = true
    WHERE is_active = true
      AND check_name IN (
          SELECT r.check_name
          FROM {FULL_SCHEMA}.check_registry r
          LEFT JOIN {FULL_SCHEMA}.check_baseline_config c
            ON r.check_name = c.check_name
          WHERE r.is_active = true
            AND UPPER(COALESCE(c.baseline_mode, 'MODEL')) = 'MODEL'
          GROUP BY r.check_name
          HAVING COUNT(*) = 1
      )
""")

# Enforce strict baseline invariant for active MODEL checks only.
active_baseline_issues = spark.sql(f"""
    SELECT
        r.check_name,
        c.baseline_mode,
        COUNT(*) AS active_row_count,
        SUM(CASE WHEN r.is_baseline = true THEN 1 ELSE 0 END) AS active_baseline_count
    FROM {FULL_SCHEMA}.check_registry r
    LEFT JOIN {FULL_SCHEMA}.check_baseline_config c
      ON r.check_name = c.check_name
    WHERE r.is_active = true
    GROUP BY r.check_name, c.baseline_mode
    HAVING UPPER(COALESCE(c.baseline_mode, 'MODEL')) = 'MODEL'
       AND SUM(CASE WHEN r.is_baseline = true THEN 1 ELSE 0 END) <> 1
""").toPandas()
if len(active_baseline_issues) > 0:
    raise RuntimeError(
        "Baseline invariant violation after add-checks: each active MODEL check_name must have exactly one active baseline row. "
        "Use data_quality_update_checks_notebook to promote a baseline.\n"
        + active_baseline_issues.head(20).to_string(index=False)
    )

auto_baseline_df = spark.sql(f"""
    SELECT check_name, model_name, workspace_id, dataset_id
    FROM {FULL_SCHEMA}.check_registry
    WHERE is_active = true
      AND is_baseline = true
      AND check_name IN (
          SELECT r.check_name
          FROM {FULL_SCHEMA}.check_registry r
          LEFT JOIN {FULL_SCHEMA}.check_baseline_config c
            ON r.check_name = c.check_name
          WHERE r.is_active = true
            AND UPPER(COALESCE(c.baseline_mode, 'MODEL')) = 'MODEL'
          GROUP BY r.check_name
          HAVING COUNT(*) = 1
      )
""").toPandas()

if len(auto_baseline_df) > 0:
    print(f"Auto-assigned baseline for {len(auto_baseline_df)} single-active-row check(s):")
    for _, row in auto_baseline_df.iterrows():
        print(f"  * {row['check_name']} -> {row['model_name']} ({row['workspace_id']}/{row['dataset_id']})")

count = spark.sql(f"SELECT COUNT(*) AS c FROM {FULL_SCHEMA}.check_registry WHERE is_active = true").collect()[0]["c"]
print(f"check_registry now has {count} active row(s)")

## View Your Checks

In [ ]:
from IPython.display import display

display(spark.sql(f"""
SELECT
    r.check_name, r.model_name, r.workspace_id, r.dataset_id,
    r.workspace_name, r.dataset_name, r.run_frequency, r.fail_delta_pct_threshold, r.is_baseline,
    COALESCE(c.baseline_mode, 'MODEL') AS baseline_mode,
    c.static_baseline_value
FROM {FULL_SCHEMA}.check_registry r
LEFT JOIN {FULL_SCHEMA}.check_baseline_config c
  ON r.check_name = c.check_name
WHERE r.is_active = true
ORDER BY r.check_name, r.model_name
""").toPandas())

print(f"\n✓  Ready to run the validation job")